# Python + NumPy Interview Practice

Practice notebook for the upcoming interview. Topics covered:
- Classes and methods
- List comprehensions
- Recursion
- `collections.namedtuple` and `typing.NamedTuple`
- Floating point comparisons
- NumPy: `np.array`, `np.min`/`np.max`, `np.sum`, `np.bincount`, `np.nonzero`, `np.where`, `np.random.RandomState`
- NumPy array broadcasting
- `unittest`

## How to use this notebook

1. Read the **concept refresher** at the top of each section.
2. Implement the problem in the cell marked `# YOUR CODE HERE`.
3. Run the test cell below it — they use `unittest` and will tell you what passes/fails.
4. If you're stuck, run the **solution** cell at the bottom of the section.

To run a cell: `Shift + Enter`. To restart kernel and clear state: Kernel → Restart.

**Debugging tip**: drop `breakpoint()` anywhere in your code to drop into pdb. Use `n` (next line), `s` (step in), `c` (continue), `p var` (print variable), `pp var` (pretty print), `l` (list code), `w` (where am I), `q` (quit).


## Setup

Run this first.

In [2]:
import sys
import numpy as np
import unittest
from typing import NamedTuple
from collections import namedtuple

def run_tests(test_case_class):
    """Helper that runs a TestCase and shows verbose output inline."""
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case_class)
    runner = unittest.TextTestRunner(verbosity=2, stream=sys.stdout)
    runner.run(suite)


---
## Concept Refreshers

Skim these before the problems. Skip what you already know cold.

### `NamedTuple` (typed version)

```python
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int
    label: str = "default"  # defaults work cleanly here

p = Point(1, 2)
p.x, p.y      # access by name
p[0]          # also indexable
p._replace(x=10)  # returns a NEW NamedTuple (they're immutable)
p._asdict()
x, y, lbl = p  # unpacking works
```

The older `collections.namedtuple` is similar but uses a factory function:
```python
from collections import namedtuple
Point = namedtuple('Point', ['x', 'y'])
```

### Recursion

```python
def factorial(n):
    if n <= 1:        # base case FIRST
        return 1
    return n * factorial(n - 1)

# Tree pattern - very common in tests:
def tree_sum(node):
    if node is None:
        return 0
    return node.value + sum(tree_sum(c) for c in node.children)
```

Default recursion limit is ~1000. Use `sys.setrecursionlimit()` or convert to iteration if needed.

### List comprehensions

```python
[x**2 for x in range(10)]
[x for x in nums if x > 0]
[(i, j) for i in range(3) for j in range(3) if i != j]  # nested
[item for sublist in nested for item in sublist]         # flatten
{k: v for k, v in pairs}      # dict comp
{x % 10 for x in nums}        # set comp
```

### NumPy: `bincount`

Counts occurrences of each non-negative integer. Output length = `arr.max() + 1` (or `minlength`).

```python
np.bincount([0, 1, 1, 3, 2, 1, 7])
# array([1, 3, 1, 1, 0, 0, 0, 1])

# With weights -> sums weights grouped by index:
np.bincount([0, 1, 1, 2], weights=[1.0, 2.0, 3.0, 4.0])
# array([1., 5., 4.])

# Use minlength to keep trailing empty bins:
np.bincount([0, 0, 1], minlength=5)
# array([2, 1, 0, 0, 0])
```

### NumPy: `RandomState`

Seeded RNG object — use this instead of `np.random.seed()` for reproducibility without global state. Tests will pass you one.

```python
rng = np.random.RandomState(42)
rng.rand(3)
rng.randint(0, 10, size=5)
rng.choice([1,2,3], size=4, replace=False)
rng.shuffle(arr)  # in-place
```

### NumPy: broadcasting

Rule: align shapes from the right, dimensions must be equal or one must be 1.

```python
a = np.array([[1, 2, 3],     # (2, 3)
              [4, 5, 6]])
b = np.array([10, 20, 30])    # (3,)
a + b   # b broadcasts across rows

# Pairwise distances pattern (memorize this!):
points = np.random.rand(N, D)
centroids = np.random.rand(K, D)
diff = points[:, None, :] - centroids[None, :, :]  # (N, K, D)
sq_dist = np.sum(diff**2, axis=-1)                  # (N, K)
```

### NumPy: `np.where` / `np.nonzero`

```python
# Two modes for np.where:
np.where(arr > 0)              # indices where True
np.where(arr > 0, arr, 0)      # element-wise if/else

# np.nonzero: indices of nonzero / truthy
np.nonzero(np.array([0, 3, 0, 5]))   # (array([1, 3]),)
```

### Floating point gotcha

Never compare floats with `==`. Use:
- `unittest.TestCase.assertAlmostEqual(a, b, places=7)` for single values
- `np.testing.assert_allclose(arr1, arr2)` for arrays
- `math.isclose(a, b)` for plain Python floats


---
## Problem 1 — Inventory tracker

Tests: classes, `typing.NamedTuple`, list comprehensions, exception handling.

Build an `Inventory` class:
- Define `Item` as a `typing.NamedTuple` with fields: `name: str`, `price: float`, `quantity: int`
- `__init__()` starts empty
- `add(item)` adds an Item, or combines quantities if the name already exists
- `remove(name, quantity)` decreases quantity; deletes the entry if it hits 0. Raises `ValueError` if the item doesn't exist or you'd go negative.
- `total_value()` returns sum of `price * quantity` across all items
- `items_below(threshold)` returns a list of item NAMES whose quantity is strictly below the threshold, sorted alphabetically. **Use a list comprehension.**


In [19]:
# YOUR CODE HERE

class Item(NamedTuple):
    name: str
    price: float
    quantity: int


class Inventory:
    def __init__(self):
        self.list = list()

    def _find(self, name):
        for i in range(len(self.list)):
            if self.list[i].name == name:
                return i
        return None

    def add(self, item):
        idx = self._find(item.name)
        if idx is not None:
            prevItem = self.list[idx]
            self.list[idx] = prevItem._replace(quantity=prevItem.quantity + item.quantity)
            return
        self.list.append(item)

    def remove(self, name, quantity):
        idx = self._find(name)
        if idx is None or self.list[idx].quantity < quantity:
             raise ValueError("Can't remove item")
        prevItem = self.list[idx]
        self.list[idx] = prevItem._replace(quantity=prevItem.quantity -  quantity)

    def total_value(self):
        tot = 0
        for i in range(len(self.list)):
            tot += self.list[i].price * self.list[i].quantity
        return tot

    def items_below(self, threshold_quantity):
        tot = []
        for i in range(len(self.list)):
            if self.list[i].quantity < threshold_quantity:
                tot.append(self.list[i].name)
        return sorted(tot)

In [ ]:
# Run the tests

class TestInventory(unittest.TestCase):
    def setUp(self):
        self.inv = Inventory()

    def test_empty(self):
        self.assertEqual(self.inv.total_value(), 0)
        self.assertEqual(self.inv.items_below(100), [])

    def test_add_and_total(self):
        self.inv.add(Item("apple", 1.50, 10))
        self.assertEqual(self.inv.total_value(), 15.0)

    def test_add_duplicate(self):
        self.inv.add(Item("apple", 1.50, 10))
        self.inv.add(Item("apple", 1.50, 5))
        self.assertEqual(self.inv.total_value(), 22.5)

    def test_remove_partial(self):
        self.inv.add(Item("apple", 1.50, 10))
        self.inv.remove("apple", 3)
        self.assertAlmostEqual(self.inv.total_value(), 10.5)

    def test_remove_to_zero(self):
        self.inv.add(Item("apple", 1.50, 10))
        self.inv.remove("apple", 10)
        self.assertEqual(self.inv.total_value(), 0)

    def test_remove_too_many_raises(self):
        self.inv.add(Item("apple", 1.50, 10))
        with self.assertRaises(ValueError):
            self.inv.remove("apple", 11)

    def test_remove_missing_raises(self):
        with self.assertRaises(ValueError):
            self.inv.remove("ghost", 1)

    def test_items_below_sorted(self):
        self.inv.add(Item("zebra", 1.0, 2))
        self.inv.add(Item("apple", 1.0, 1))
        self.inv.add(Item("mango", 1.0, 50))
        self.assertEqual(self.inv.items_below(5), ["apple", "zebra"])

run_tests(TestInventory)


test_add_and_total (__main__.TestInventory.test_add_and_total) ... ok
test_add_duplicate (__main__.TestInventory.test_add_duplicate) ... ok
test_empty (__main__.TestInventory.test_empty) ... ok
test_items_below_sorted (__main__.TestInventory.test_items_below_sorted) ... ok
test_remove_missing_raises (__main__.TestInventory.test_remove_missing_raises) ... ok
test_remove_partial (__main__.TestInventory.test_remove_partial) ... ok
test_remove_to_zero (__main__.TestInventory.test_remove_to_zero) ... ok
test_remove_too_many_raises (__main__.TestInventory.test_remove_too_many_raises) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.006s

OK


<details><summary><b>Click for solution</b></summary>

Uncomment the cell below and run it.

In [ ]:
# Solution (uncomment to run)

# class Item(NamedTuple):
#     name: str
#     price: float
#     quantity: int
#
# class Inventory:
#     def __init__(self):
#         self._items = {}
#
#     def add(self, item):
#         if item.name in self._items:
#             existing = self._items[item.name]
#             self._items[item.name] = existing._replace(
#                 quantity=existing.quantity + item.quantity
#             )
#         else:
#             self._items[item.name] = item
#
#     def remove(self, name, quantity):
#         if name not in self._items:
#             raise ValueError(f"Item {name!r} not in inventory")
#         existing = self._items[name]
#         new_qty = existing.quantity - quantity
#         if new_qty < 0:
#             raise ValueError(f"Cannot remove {quantity}; only {existing.quantity} available")
#         if new_qty == 0:
#             del self._items[name]
#         else:
#             self._items[name] = existing._replace(quantity=new_qty)
#
#     def total_value(self):
#         return sum(item.price * item.quantity for item in self._items.values())
#
#     def items_below(self, threshold_quantity):
#         return sorted([
#             item.name
#             for item in self._items.values()
#             if item.quantity < threshold_quantity
#         ])


---
## Problem 2 — Directory tree (recursion)

Best time: 25 mins

Tests: recursion, NamedTuple, `isinstance`, list comprehensions for flattening.

Model a filesystem. Each node is either a `File` (with `size` in bytes) or a `Directory` (with `children`, a tuple of Files/Directories).

Implement recursively:
- `total_size(node)` — sum of all file sizes under this node
- `file_count(node)` — count files only (not directories)
- `find_files(node, min_size)` — list of file names with `size >= min_size`, any order
- `deepest_path(node)` — list of names from root to deepest leaf. On ties, return the depth-first left-to-right first.


In [45]:
# YOUR CODE HERE

class File(NamedTuple):
    name: str
    size: int

class Directory(NamedTuple):
    name: str
    children: tuple  # of File or Directory

def _dfs(node, on_leaf, path=[]):
    if hasattr(node, "size"): # leaf
        return on_leaf(node, path + [node.name])
    for c in node.children:
        _dfs(c, on_leaf, path = path + [node.name])

def total_size(node):
    tot = [0]
    def on_leaf(n, p):
        tot[0] += n.size
    _dfs(node, on_leaf=on_leaf)
    return tot[0]

def file_count(node):
    tot = [0]
    def on_leaf(n, p):
        tot[0] += 1
    _dfs(node, on_leaf=on_leaf)
    return tot[0]


def find_files(node, min_size):
    files = []
    def on_leaf(n, p):
        if n.size >=min_size:
            files.append(p)
    _dfs(node, on_leaf=on_leaf)
    return [f[-1] for f in files]

def deepest_path(node):
    max_depth_num = [[]]
    def on_leaf(n, p):
        if len(p) > len(max_depth_num[0]):
            max_depth_num[0] = p
    _dfs(node, on_leaf=on_leaf)
    return max_depth_num[0]

In [46]:
# Run the tests

class TestTree(unittest.TestCase):
    def setUp(self):
        self.tree = Directory("root", (
            Directory("docs", (
                File("readme.md", 100),
                File("notes.txt", 50),
            )),
            Directory("src", (
                Directory("app", (
                    File("main.py", 500),
                    File("util.py", 200),
                )),
                File("test.py", 75),
            )),
            Directory("empty_dir", ()),
        ))

    def test_total_size(self):
        self.assertEqual(total_size(self.tree), 925)

    def test_total_size_file(self):
        self.assertEqual(total_size(File("x", 42)), 42)

    def test_file_count(self):
        self.assertEqual(file_count(self.tree), 5)

    def test_file_count_empty(self):
        self.assertEqual(file_count(Directory("empty", ())), 0)

    def test_find_files(self):
        result = find_files(self.tree, min_size=100)
        self.assertEqual(sorted(result), ["main.py", "readme.md", "util.py"])

    def test_find_none(self):
        self.assertEqual(find_files(self.tree, min_size=10000), [])

    def test_deepest_path(self):
        self.assertEqual(deepest_path(self.tree), ["root", "src", "app", "main.py"])

run_tests(TestTree)


test_deepest_path (__main__.TestTree.test_deepest_path) ... ok
test_file_count (__main__.TestTree.test_file_count) ... ok
test_file_count_empty (__main__.TestTree.test_file_count_empty) ... ok
test_find_files (__main__.TestTree.test_find_files) ... ok
test_find_none (__main__.TestTree.test_find_none) ... ok
test_total_size (__main__.TestTree.test_total_size) ... ok
test_total_size_file (__main__.TestTree.test_total_size_file) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.005s

OK


<details><summary><b>Click for solution</b></summary>

In [ ]:
# Solution

# def total_size(node):
#     if isinstance(node, File):
#         return node.size
#     return sum(total_size(c) for c in node.children)
#
# def file_count(node):
#     if isinstance(node, File):
#         return 1
#     return sum(file_count(c) for c in node.children)
#
# def find_files(node, min_size):
#     if isinstance(node, File):
#         return [node.name] if node.size >= min_size else []
#     return [n for c in node.children for n in find_files(c, min_size)]
#
# def deepest_path(node):
#     if isinstance(node, File):
#         return [node.name]
#     if not node.children:
#         return [node.name]
#     best = []
#     for c in node.children:
#         candidate = deepest_path(c)
#         if len(candidate) > len(best):
#             best = candidate
#     return [node.name] + best


---
## Problem 3 — KMeansStep (NumPy broadcasting + bincount + RandomState)

Tests: NumPy broadcasting, `np.bincount` with `minlength`, `np.random.RandomState`, `np.argmin`.

One iteration of k-means. Build a `KMeansStep` class:

- `__init__(n_clusters, rng)` — store both. `rng` is a `np.random.RandomState`.
- `initialize_centroids(points)` — `points` is shape `(N, D)`. Pick `n_clusters` distinct rows using `self.rng.choice` with `replace=False`. Return shape `(n_clusters, D)`.
- `assign(points, centroids)` — for each point, return index of nearest centroid (Euclidean). Return shape `(N,)`. **Use broadcasting.**
- `cluster_sizes(assignments)` — return shape `(n_clusters,)` of counts. Empty clusters must show as 0. **Use `np.bincount` with `minlength`.**
- `update_centroids(points, assignments)` — return new centroids: mean of points per cluster. Empty clusters → zeros. Shape `(n_clusters, D)`.


In [76]:
# YOUR CODE HERE

class KMeansStep:
    def __init__(self, n_clusters, rng):
        self.n_clusters = n_clusters
        self.rng = rng

    def initialize_centroids(self, points):
        ids = np.arange(len(points))
        ids = self.rng.choice(ids, self.n_clusters)
        return points[ids]

    def assign(self, points, centroids):
        # (N, D), (K, D) -> (N,)
        diffs = np.linalg.norm(np.expand_dims(points, axis=1) - np.expand_dims(centroids, axis=0), axis=-1) # (N, K)
        return np.argmin(diffs, axis=-1)

    def cluster_sizes(self, assignments):
        return np.bincount(assignments, minlength=self.n_clusters)

    def update_centroids(self, points, assignments):
        # (N, D), (N,)
        N, D = points.shape
        cluster_counts = np.bincount(assignments, minlength=self.n_clusters)
        cluster_sums = np.zeros((self.n_clusters, D))
        for k in range(self.n_clusters):
            cluster_sums[k] = np.sum(points[assignments == k], axis=0) # (K, D) -> (D,)
        return cluster_sums / np.expand_dims(np.clip(cluster_counts, a_max=None, a_min=1), -1)


# rng = np.random.RandomState(42)
# np.expand_dims(rng.rand(3), axis=1).shape
# rng.randint(0, 10, size=5)
# rng.choice([1,2,3], size=2, replace=False)
# rng.shuffle(arr)  # in-place


In [78]:
# Run the tests

class TestKMeansStep(unittest.TestCase):
    def test_init_shape(self):
        km = KMeansStep(3, np.random.RandomState(0))
        points = np.arange(20).reshape(10, 2).astype(float)
        self.assertEqual(km.initialize_centroids(points).shape, (3, 2))

    def test_init_distinct(self):
        km = KMeansStep(4, np.random.RandomState(42))
        points = np.arange(40).reshape(20, 2).astype(float)
        c = km.initialize_centroids(points)
        self.assertEqual(np.unique(c, axis=0).shape[0], 4)

    def test_init_reproducible(self):
        points = np.arange(40).reshape(20, 2).astype(float)
        c1 = KMeansStep(3, np.random.RandomState(7)).initialize_centroids(points)
        c2 = KMeansStep(3, np.random.RandomState(7)).initialize_centroids(points)
        np.testing.assert_array_equal(c1, c2)

    def test_assign_simple(self):
        km = KMeansStep(2, np.random.RandomState(0))
        points = np.array([[0.0, 0.0], [0.1, 0.0], [10.0, 10.0], [10.1, 10.0]])
        centroids = np.array([[0.0, 0.0], [10.0, 10.0]])
        np.testing.assert_array_equal(km.assign(points, centroids), [0, 0, 1, 1])

    def test_cluster_sizes_with_empty(self):
        km = KMeansStep(4, np.random.RandomState(0))
        assignments = np.array([0, 0, 0, 1, 2, 2])
        np.testing.assert_array_equal(
            km.cluster_sizes(assignments), [3, 1, 2, 0]
        )

    def test_update_basic(self):
        km = KMeansStep(2, np.random.RandomState(0))
        points = np.array([[0.0, 0.0], [2.0, 0.0], [10.0, 10.0], [12.0, 10.0]])
        assignments = np.array([0, 0, 1, 1])
        np.testing.assert_allclose(
            km.update_centroids(points, assignments),
            [[1.0, 0.0], [11.0, 10.0]]
        )

    def test_update_empty_is_zero(self):
        km = KMeansStep(3, np.random.RandomState(0))
        points = np.array([[1.0, 1.0], [3.0, 3.0]])
        assignments = np.array([0, 2])
        np.testing.assert_allclose(
            km.update_centroids(points, assignments),
            [[1.0, 1.0], [0.0, 0.0], [3.0, 3.0]]
        )

run_tests(TestKMeansStep)


test_assign_simple (__main__.TestKMeansStep.test_assign_simple) ... ok
test_cluster_sizes_with_empty (__main__.TestKMeansStep.test_cluster_sizes_with_empty) ... ok
test_init_distinct (__main__.TestKMeansStep.test_init_distinct) ... ok
test_init_reproducible (__main__.TestKMeansStep.test_init_reproducible) ... ok
test_init_shape (__main__.TestKMeansStep.test_init_shape) ... ok
test_update_basic (__main__.TestKMeansStep.test_update_basic) ... ok
test_update_empty_is_zero (__main__.TestKMeansStep.test_update_empty_is_zero) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.009s

OK


<details><summary><b>Click for solution</b></summary>

In [ ]:
# Solution

# class KMeansStep:
#     def __init__(self, n_clusters, rng):
#         self.n_clusters = n_clusters
#         self.rng = rng
#
#     def initialize_centroids(self, points):
#         idx = self.rng.choice(points.shape[0], size=self.n_clusters, replace=False)
#         return points[idx]
#
#     def assign(self, points, centroids):
#         # points (N,D), centroids (K,D)
#         diff = points[:, None, :] - centroids[None, :, :]  # (N, K, D)
#         sq_dist = np.sum(diff ** 2, axis=-1)               # (N, K)
#         return np.argmin(sq_dist, axis=1)                  # (N,)
#
#     def cluster_sizes(self, assignments):
#         return np.bincount(assignments, minlength=self.n_clusters)
#
#     def update_centroids(self, points, assignments):
#         D = points.shape[1]
#         new = np.zeros((self.n_clusters, D))
#         for k in range(self.n_clusters):
#             mask = assignments == k
#             if mask.any():
#                 new[k] = points[mask].mean(axis=0)
#         return new


---
## Problem 4 — NumPy masking drill

Tests: `np.where` (both modes), `np.nonzero`, `np.bincount` with `weights=`.

Implement four small functions. Each is one or two lines if you know the right NumPy idiom.

- `clip_negatives(arr)` — replace negative values with 0, keep positives. Use `np.where`.
- `positive_indices(arr)` — return a 1D array of indices where `arr > 0`. Use `np.nonzero` or `np.where`.
- `replace_above(arr, threshold, replacement)` — where `arr > threshold`, replace with `replacement`; otherwise keep. Use `np.where`.
- `group_means(values, groups, n_groups)` — `values` is float `(N,)`, `groups` is int `(N,)` in `[0, n_groups)`. Return mean of values per group; empty groups → 0. **Use `np.bincount` with `weights=` for the sums.**


In [117]:
# YOUR CODE HERE

def clip_negatives(arr):
    return np.where(arr >= 0, arr, np.zeros_like(arr))


def positive_indices(arr):
    pos = np.nonzero(np.where(arr > 0, arr, np.zeros_like(arr)))
    return pos[0]


def replace_above(arr, threshold, replacement):
    narr = np.where(arr <= threshold, arr, np.ones_like(arr) * replacement)
    # print(f"***{narr}")
    return narr 

def group_means(values, groups, n_groups):
    tot_sum = np.bincount(groups, weights=values, minlength=n_groups)
    tot_count = np.bincount(groups, minlength=n_groups)
    return tot_sum / np.where(tot_count>0, tot_count, np.ones_like(tot_count))

In [118]:
np.nonzero(np.array([0, -1, 0, -2, 3]) ** 2)

(array([1, 3, 4]),)

In [119]:
# Run the tests

class TestNumpyDrill(unittest.TestCase):
    def test_clip_negatives(self):
        a = np.array([-1.0, 2.0, -3.0, 4.0, 0.0])
        np.testing.assert_array_equal(clip_negatives(a), [0, 2, 0, 4, 0])

    def test_positive_indices(self):
        a = np.array([0, -1, 5, -2, 3])
        np.testing.assert_array_equal(positive_indices(a), [2, 4])

    def test_replace_above(self):
        a = np.array([1, 5, 10, 15, 20])
        np.testing.assert_array_equal(replace_above(a, 10, -1), [1, 5, 10, -1, -1])

    def test_group_means_basic(self):
        values = np.array([1.0, 3.0, 5.0, 7.0])
        groups = np.array([0, 0, 1, 1])
        np.testing.assert_allclose(group_means(values, groups, 2), [2.0, 6.0])

    def test_group_means_with_empty(self):
        values = np.array([10.0, 20.0])
        groups = np.array([0, 2])
        np.testing.assert_allclose(group_means(values, groups, 3), [10.0, 0.0, 20.0])

run_tests(TestNumpyDrill)


test_clip_negatives (__main__.TestNumpyDrill.test_clip_negatives) ... ok
test_group_means_basic (__main__.TestNumpyDrill.test_group_means_basic) ... ok
test_group_means_with_empty (__main__.TestNumpyDrill.test_group_means_with_empty) ... ok
test_positive_indices (__main__.TestNumpyDrill.test_positive_indices) ... ok
test_replace_above (__main__.TestNumpyDrill.test_replace_above) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.004s

OK


<details><summary><b>Click for solution</b></summary>

In [ ]:
# Solution

# def clip_negatives(arr):
#     return np.where(arr < 0, 0, arr)
#
# def positive_indices(arr):
#     return np.nonzero(arr > 0)[0]
#     # equivalent: np.where(arr > 0)[0]
#
# def replace_above(arr, threshold, replacement):
#     return np.where(arr > threshold, replacement, arr)
#
# def group_means(values, groups, n_groups):
#     sums = np.bincount(groups, weights=values, minlength=n_groups)
#     counts = np.bincount(groups, minlength=n_groups)
#     means = np.zeros(n_groups)
#     nonzero = counts > 0
#     means[nonzero] = sums[nonzero] / counts[nonzero]
#     return means


---
## Scratchpad

Use the cells below to experiment, try out NumPy operations, test recursion patterns, etc. The interview itself will likely involve building intuition under pressure — practice exploring here.

A few things worth trying interactively:
- `np.bincount` with and without `minlength` and `weights`
- Pairwise distance via broadcasting on 3D arrays
- `np.where` in both modes — single-arg vs three-arg
- `NamedTuple._replace` on nested structures


In [ ]:
# experiment freely
